# Cross-Dataset Predictive Maintenance of Fluid-Power Systems

This project investigates whether common pre-failure sensor patterns can be identified across two independent fluid-power systems: a metro train air compressor (MetroPT-3) and a hydraulic test rig (Condition Monitoring of Hydraulic Systems). Both datasets describe physically comparable machinery but differ in scale, environment, and monitoring method, which makes their combined analysis a useful test of whether degradation signatures generalize across different types of fluid-power equipment.

The work is structured across four notebooks: data loading and cleaning (01), exploratory data analysis (02), hypothesis testing and reliability-metric analysis (03), and a final cross-dataset comparison and conclusions (04). This notebook focuses on the first stage: loading both datasets from their original sources and preparing them for consistent downstream analysis.

---

# **Data Loading and Cleaning**

## Dataset context and selection

Two independent datasets related to predictive maintenance in fluid-power systems are used here. They were chosen because they represent complementary engineering settings: one is a real operational compressor monitored continuously over time, while the other is a controlled hydraulic test rig measured cycle by cycle under known degradation conditions.

### Dataset 1: MetroPT-3

The MetroPT-3 dataset originates from a predictive maintenance study on the urban metro transportation system in Porto, Portugal. A signal acquisition system was installed on the Air Production Unit (APU) of a real operational metro train. The APU supplies compressed air to pneumatic subsystems such as doors and brakes, which makes its reliability operationally important.

The dataset contains approximately seven months of continuous multivariate sensor data, recorded from February to September 2020, with a nominal sampling interval of 10 seconds. The variables include analog measurements such as pressure, oil temperature, and motor current, together with digital signals describing operating states, valve conditions, and alarm-like events.

This dataset is particularly useful because it reflects real industrial operation rather than a laboratory simulation. As a result, it contains realistic state changes, interruptions, regime shifts, and failure-related behavior. Although the time series itself is unlabeled, accompanying failure reports document four incidents during the monitoring period, all related to air leaks of varying severity. This makes MetroPT-3 well suited for studying temporal behavior, operating-regime changes, and potential early warning patterns in a real compressor system.

**Source:** Davari, N., Veloso, B., Ribeiro, R., & Gama, J. (2021). *MetroPT-3 Dataset.* UCI Machine Learning Repository.

### Dataset 2: Condition Monitoring of Hydraulic Systems

The second dataset, Condition Monitoring of Hydraulic Systems, was collected from a controlled hydraulic test rig at Saarland University, Germany. Unlike MetroPT-3, which captures a real machine in continuous operation, this dataset was produced under laboratory conditions designed for systematic condition monitoring.

The hydraulic system executes repeated 60-second load cycles, during which pressures, flow rates, temperatures, vibration, motor power, and efficiency-related quantities are measured. In total, the dataset contains 2205 operating cycles. Four main hydraulic components (cooler, valve, pump, and accumulator), were varied across different degradation levels, and an additional `stable_flag` indicates whether the system had reached stable behavior during a given cycle.

This dataset is useful because it provides structured condition labels and controlled degradation states, which are not directly available in MetroPT-3. The raw per-cycle sensor matrices are measured at sampling rates between 1 Hz and 100 Hz depending on sensor type. In this project, these raw cycle signals are reduced to summary statistics (mean, standard deviation, minimum, and maximum) for each sensor, resulting in a compact feature table that can be analyzed cycle by cycle.

**Source:** Helwig, N., Pignanelli, E., & Schütze, A. (2015). *Condition Monitoring of Hydraulic Systems.* UCI Machine Learning Repository.

### Why this combination is useful

The two datasets differ substantially in structure, but this difference is a strength rather than a weakness. MetroPT-3 represents long-term monitoring of a real compressor system under operational conditions, while the hydraulic dataset represents controlled cycle-based monitoring with explicit degradation labels.

Together, they support a broader predictive-maintenance perspective. MetroPT-3 makes it possible to study evolving behavior over time in a real machine, while the hydraulic dataset makes it possible to study how sensor-derived features vary across known condition states. The common link is that both describe fluid-power machinery monitored through physically meaningful sensor signals such as pressure, temperature, electrical load or power, and state-related variables. This makes them suitable for one coherent cross-dataset analysis.

## Imports

In [17]:
import pandas as pd
import numpy as np

## MetroPT-3 Air Compressor Dataset (Porto, Portugal)

In [2]:
metro = pd.read_csv('../data/raw/MetroPT-3 Train Dataset/MetroPT3(AirCompressor).csv')
metro.head()

,Unnamed: 0,timestamp,TP2,TP3,H1,DV_pressure,Reservoirs,Oil_temperature,Motor_current,COMP,DV_eletric,Towers,MPG,LPS,Pressure_switch,Oil_level,Caudal_impulses
0,0,2020-02-01 00:00:00,-0.012,9.358,9.340,-0.024,9.358,53.600,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
1,10,2020-02-01 00:00:10,-0.014,9.348,9.332,-0.022,9.348,53.675,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
2,20,2020-02-01 00:00:19,-0.012,9.338,9.322,-0.022,9.338,53.600,0.0425,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
3,30,2020-02-01 00:00:29,-0.012,9.328,9.312,-0.022,9.328,53.425,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
4,40,2020-02-01 00:00:39,-0.012,9.318,9.302,-0.022,9.318,53.475,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0


### MetroPT variable description

The MetroPT-3 dataset contains sensor readings from an air compressor on a metro train in Porto, Portugal. The sensors record the following:

- **TP2** - Pressure on the compressor (bar)
- **TP3** - Pressure at the pneumatic panel (bar)
- **H1** - Pressure drop at the cyclonic separator filter (bar)
- **DV_pressure** - Pressure drop when towers discharge air dryers (bar); zero indicates compressor operating under load
- **Reservoirs** - Downstream pressure of the reservoirs (bar); should be close to TP3
- **Oil_temperature** - Oil temperature on the compressor (°C)
- **Motor_current** - Current of one phase of the three-phase motor (A); it presents values close to 0A - when it turns off, 4A - when working offloaded, 7A - when working under load, and 9A - when it starts working
- **COMP** - Air intake valve; active when compressor is OFF or offloaded
- **DV_eletric** - Compressor outlet valve; active under load, inactive when off or offloaded
- **Towers** - Air drying tower selector; 0 = tower one active, 1 = tower two active
- **MPG** - Starts compressor under load when pressure drops below 8.2 bar
- **LPS** - Low pressure alarm; activates when pressure drops below 7 bar
- **Pressure_switch** - Detects discharge in air-drying towers
- **Oil_level** - Active when oil is below expected levels
- **Caudal_impulses** - Counts air flow pulses from APU to reservoirs

*Source: MetroPT-3 Dataset, UCI Machine Learning Repository*  
*https://archive.ics.uci.edu/dataset/791/metropt-3+dataset*

In [3]:
# Fix timestamp
metro['timestamp'] = pd.to_datetime(metro['timestamp'])

# Drop useless column
metro = metro.drop(columns=['Unnamed: 0'])

metro.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1516948 entries, 0 to 1516947
Data columns (total 16 columns):
 #   Column           Non-Null Count    Dtype         
---  ------           --------------    -----         
 0   timestamp        1516948 non-null  datetime64[ns]
 1   TP2              1516948 non-null  float64       
 2   TP3              1516948 non-null  float64       
 3   H1               1516948 non-null  float64       
 4   DV_pressure      1516948 non-null  float64       
 5   Reservoirs       1516948 non-null  float64       
 6   Oil_temperature  1516948 non-null  float64       
 7   Motor_current    1516948 non-null  float64       
 8   COMP             1516948 non-null  float64       
 9   DV_eletric       1516948 non-null  float64       
 10  Towers           1516948 non-null  float64       
 11  MPG              1516948 non-null  float64       
 12  LPS              1516948 non-null  float64       
 13  Pressure_switch  1516948 non-null  float64       
 14  Oi

### Initial Cleaning

The raw dataset contained an unnecessary auto-generated index column (`Unnamed: 0`) which was removed. The `timestamp` column was converted from string to `datetime64` format to enable time-based analysis.

In [63]:
# Checking for duplicates
duplicate_rows = metro.duplicated().sum()
duplicate_timestamps = metro['timestamp'].duplicated().sum()

print(f"Duplicate rows: {duplicate_rows}")
print(f"Duplicate timestamps: {duplicate_timestamps}")

Duplicate rows: 0
Duplicate timestamps: 0


### Observations

After initial cleaning, the dataset contains **1,516,948 records** and **16 columns** with no missing values. No duplicate rows or timestamps were found, confirming the data is clean and suitable for time-based analysis.

### Sort by timestamp and check date range

This will tell us exactly what time period the dataset covers:

In [5]:
metro = metro.sort_values('timestamp').reset_index(drop=True)

print(f"Date range: {metro['timestamp'].min()} → {metro['timestamp'].max()}")
print(f"Total duration: {metro['timestamp'].max() - metro['timestamp'].min()}")

Date range: 2020-02-01 00:00:00 → 2020-09-01 03:59:50
Total duration: 213 days 03:59:50


In [6]:
metro['time_diff'] = metro['timestamp'].diff()

print("Most common intervals:")
print(metro['time_diff'].value_counts().head(10))

print("\nInterval statistics:")
print(metro['time_diff'].describe())

Most common intervals:
time_diff
0 days 00:00:10    1337521
0 days 00:00:09     128277
0 days 00:00:12      38321
0 days 00:00:13       7988
0 days 00:00:11       4471
0 days 00:00:21         10
0 days 00:00:19          5
0 days 00:00:22          4
0 days 00:00:14          3
0 days 00:00:17          3
Name: count, dtype: int64

Interval statistics:
count                      1516947
mean     0 days 00:00:12.141221809
std      0 days 00:05:14.107266185
min                0 days 00:00:08
25%                0 days 00:00:10
50%                0 days 00:00:10
75%                0 days 00:00:10
max                2 days 00:01:58
Name: time_diff, dtype: object


### Observations

The temporal structure of the MetroPT dataset is close to regular, with a dominant sampling interval of 10 seconds and a median interval of 10 seconds as well. Small variations of 9-13 seconds are also present, which suggests minor timing irregularities typical of real-world sensor systems. In addition, the dataset contains a few much larger gaps, including interruptions of up to two days, which indicates that the recording is not perfectly continuous over the full monitoring period.

Checking largest gaps:

In [7]:
large_gaps = metro[metro['time_diff'] > pd.Timedelta(minutes=5)][['timestamp', 'time_diff']]
print(f"Number of gaps > 5 minutes: {len(large_gaps)}")
print("\nLargest gaps:")
print(large_gaps.nlargest(10, 'time_diff'))

Number of gaps > 5 minutes: 275

Largest gaps:
                  timestamp       time_diff
618538  2020-04-27 01:12:49 2 days 00:01:58
1055664 2020-06-28 23:07:43 1 days 12:14:36
214850  2020-03-01 04:00:09 1 days 04:03:01
1318888 2020-08-05 08:23:01 1 days 00:40:33
805415  2020-05-25 01:14:14 1 days 00:34:51
1126187 2020-07-08 15:20:51 0 days 23:56:00
1454438 2020-08-23 18:51:01 0 days 23:39:17
708881  2020-05-10 22:48:58 0 days 22:17:41
908125  2020-06-08 11:48:04 0 days 21:28:25
450114  2020-04-02 09:59:17 0 days 20:43:37


In [8]:
metro = metro.drop(columns=['time_diff'])

A total of 275 gaps longer than 5 minutes were detected. 
The presence of larger temporal gaps may indicate periods when the monitored system was offline, under maintenance, or temporarily removed from operation. This interpretation is consistent with the run-to-failure character of the dataset, although the exact cause of each interruption cannot be confirmed from the sensor table alone.

**Data trustworthiness**

No duplicated rows or timestamps were found. The dominant sampling interval is 10 seconds, with minor variation (9-13 seconds) typical for industrial sensors. 275 interruptions longer than 5 minutes were identified and interpreted as likely maintenance periods. The data is judged reliable for downstream analysis, with the temporal gaps acknowledged when interpreting time-based patterns.

### Continuous vs Binary Columns

The documentation confirms that several 0/1 variables in MetroPT are not ordinary sensor measurements, but electrical or control signals describing machine state, alarms, and operating conditions. This supports treating them separately from continuous pressure, temperature, and current measurements during the analysis.

In [9]:
binary_cols = [
    col for col in metro.columns
    if col != 'timestamp' and metro[col].nunique(dropna=True) == 2
]

continuous_cols = [
    col for col in metro.columns
    if col != 'timestamp' and col not in binary_cols
]

print(f"Binary/state columns ({len(binary_cols)}): {binary_cols}")
print(f"Continuous columns ({len(continuous_cols)}): {continuous_cols}")

Binary/state columns (8): ['COMP', 'DV_eletric', 'Towers', 'MPG', 'LPS', 'Pressure_switch', 'Oil_level', 'Caudal_impulses']
Continuous columns (7): ['TP2', 'TP3', 'H1', 'DV_pressure', 'Reservoirs', 'Oil_temperature', 'Motor_current']


In [10]:
for col in binary_cols:
    print(f"{col}: {sorted(metro[col].unique())}")

COMP: [np.float64(0.0), np.float64(1.0)]
DV_eletric: [np.float64(0.0), np.float64(1.0)]
Towers: [np.float64(0.0), np.float64(1.0)]
MPG: [np.float64(0.0), np.float64(1.0)]
LPS: [np.float64(0.0), np.float64(1.0)]
Pressure_switch: [np.float64(0.0), np.float64(1.0)]
Oil_level: [np.float64(0.0), np.float64(1.0)]
Caudal_impulses: [np.float64(0.0), np.float64(1.0)]


In [11]:
binary_cols = [
    'COMP', 'DV_eletric', 'Towers', 'MPG',
    'LPS', 'Pressure_switch', 'Oil_level', 'Caudal_impulses'
]

continuous_cols = [
    'TP2', 'TP3', 'H1', 'DV_pressure',
    'Reservoirs', 'Oil_temperature', 'Motor_current'
]

### Observations

The MetroPT variables were separated into continuous sensor measurements and binary-valued state-like variables. Automatic detection based on the number of unique values identified eight binary-valued columns, all of which were confirmed to contain only 0/1 values.

In [12]:
# continuous summary

metro[continuous_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
TP2,1516948.0,1.367826,3.250930,-0.032,-0.014,-0.012,-0.0100,10.676
TP3,1516948.0,8.984611,0.639095,0.730,8.492,8.960,9.4920,10.302
H1,1516948.0,7.568155,3.333200,-0.036,8.254,8.784,9.3740,10.288
DV_pressure,1516948.0,0.055956,0.382402,-0.032,-0.022,-0.020,-0.0180,9.844
Reservoirs,1516948.0,8.985233,0.638307,0.712,8.494,8.960,9.4920,10.300
Oil_temperature,1516948.0,62.644182,6.516261,15.400,57.775,62.700,67.2500,89.050
Motor_current,1516948.0,2.050171,2.302053,0.020,0.040,0.045,3.8075,9.295


In [13]:
# binary frequency tables

binary_summary = pd.DataFrame({
    col: metro[col].value_counts(normalize=True)
    for col in binary_cols
}).T.fillna(0)

In [14]:
binary_summary

,0.0,1.0
COMP,0.163043,0.836957
DV_eletric,0.839389,0.160611
Towers,0.080152,0.919848
MPG,0.167336,0.832664
LPS,0.996580,0.003420
Pressure_switch,0.008563,0.991437
Oil_level,0.095844,0.904156
Caudal_impulses,0.062893,0.937107


The binary-valued variables are strongly imbalanced, with several state indicators spending most of the time in a single dominant state. Notably, COMP=1 (83.7%) indicates the compressor is OFF or offloaded, not actively compressing, suggesting it operates in short active bursts. LPS (0.34% active) and Pressure_switch (0.86% active) function as alarm signals, triggering only under abnormal conditions such as pressure drops - making them particularly 
relevant for pre-failure detection.

In [15]:
metro.to_csv('../data/processed/metro_cleaned.csv', index=False)
print("Saved successfully!")

Saved successfully!


In [16]:
# Verify saved file
metro_check = pd.read_csv('../data/processed/metro_cleaned.csv')
print(metro_check.shape)
print(metro_check.columns.tolist())
print(metro_check.head(2))

(1516948, 16)
['timestamp', 'TP2', 'TP3', 'H1', 'DV_pressure', 'Reservoirs', 'Oil_temperature', 'Motor_current', 'COMP', 'DV_eletric', 'Towers', 'MPG', 'LPS', 'Pressure_switch', 'Oil_level', 'Caudal_impulses']
             timestamp    TP2    TP3     H1  DV_pressure  Reservoirs  \
0  2020-02-01 00:00:00 -0.012  9.358  9.340       -0.024       9.358   
1  2020-02-01 00:00:10 -0.014  9.348  9.332       -0.022       9.348   

   Oil_temperature  Motor_current  COMP  DV_eletric  Towers  MPG  LPS  \
0           53.600           0.04   1.0         0.0     1.0  1.0  0.0   
1           53.675           0.04   1.0         0.0     1.0  1.0  0.0   

   Pressure_switch  Oil_level  Caudal_impulses  
0              1.0        1.0              1.0  
1              1.0        1.0              1.0  


## Hydraulic System Dataset

This dataset contains cycle-based sensor measurements from a hydraulic test rig. Unlike the MetroPT-3 dataset, which records continuous operational data over time, the hydraulic dataset is organized by repeated working cycles.

Each row corresponds to one 60-second operating cycle. The sensors were sampled at different frequencies depending on the measured physical quantity:

- Pressure sensors (`PS1`-`PS6`) and motor power (`EPS1`): 100 Hz
- Volume flow sensors (`FS1`, `FS2`): 10 Hz
- Temperature sensors (`TS1`-`TS4`), vibration (`VS1`), and virtual efficiency signals (`CE`, `CP`, `SE`): 1 Hz

The condition labels are stored separately in `profile.txt`, where each row corresponds to the same cycle number as the sensor matrices. The labels describe degradation states of several hydraulic components:

- `Cooler_condition`
- `Valve_condition`
- `Pump_leakage`
- `Accumulator_pressure`
- `Stable_flag`

The first four represent component condition or degradation severity, while `Stable_flag` indicates whether steady-state conditions were reached during the cycle.

*Source: Condition Monitoring of Hydraulic Systems, UCI Machine Learning Repository*  
*https://archive.ics.uci.edu/dataset/447/condition+monitoring+of+hydraulic+systems*

In [19]:
hydraulics_path = '../data/raw/Predictive Maintenance Of Hydraulics System/'

profile = pd.read_csv(hydraulics_path + 'profile.txt', 
                      sep='\t', 
                      header=None,
                      names=['cooler_condition', 'valve_condition', 'pump_leakage', 'accumulator_pressure', 'stable_flag'])

print(profile.shape)
profile.head()

(2205, 5)


,cooler_condition,valve_condition,pump_leakage,accumulator_pressure,stable_flag
0,3,100,0,130,1
1,3,100,0,130,1
2,3,100,0,130,1
3,3,100,0,130,1
4,3,100,0,130,1


#### Condition distribution summary (count and proportion)

Column headers represent condition values. Each cell shows the number of cycles and the corresponding proportion for a given degradation state. Empty cells indicate that the condition value does not apply to that component.

In [53]:
summary_counts = profile.apply(lambda col: col.value_counts().sort_index()).T

summary_counts_display = summary_counts.fillna(0).astype(int)
summary_counts_display = summary_counts_display.replace(0, '')

summary_counts_display

,0,1,2,3,20,73,80,90,100,115,130
cooler_condition,,,,732,732,,,,741,,
valve_condition,,,,,,360,360,360,1125,,
pump_leakage,1221,492,492,,,,,,,,
accumulator_pressure,,,,,,,,808,399,399,599
stable_flag,1449,756,,,,,,,,,


In [38]:
summary_props = profile.apply(lambda col: col.value_counts(normalize=True).sort_index()).T.round(3)

summary_props_display = summary_props.fillna('')
summary_props_display

,0,1,2,3,20,73,80,90,100,115,130
cooler_condition,,,,0.332,0.332,,,,0.336,,
valve_condition,,,,,,0.163,0.163,0.163,0.51,,
pump_leakage,0.554,0.223,0.223,,,,,,,,
accumulator_pressure,,,,,,,,0.366,0.181,0.181,0.272
stable_flag,0.657,0.343,,,,,,,,,


### Loading Sensor Data

To understand sensor behavior across different degradation states, a representative selection of sensors is loaded first.

**Pressure sensor: PS1**

In [54]:
PS1 = pd.read_csv(hydraulics_path + 'PS1.txt', sep='\t', header=None)

print(PS1.shape)
PS1.head()

(2205, 6000)


,0,1,2,3,4,5,6,7,8,9,...,5990,5991,5992,5993,5994,5995,5996,5997,5998,5999
0,151.47,151.45,151.52,151.27,150.80,150.69,153.89,154.67,152.88,153.82,...,151.16,151.19,151.25,151.16,151.10,151.16,151.14,151.10,151.21,151.19
1,151.11,151.12,151.16,150.92,150.70,150.62,152.40,153.21,152.81,153.53,...,150.82,150.82,150.86,150.80,150.73,150.79,150.84,150.79,150.80,150.86
2,150.81,150.79,150.84,150.65,150.35,150.23,152.03,152.81,152.44,153.27,...,150.49,150.44,150.47,150.46,150.38,150.47,150.50,150.43,150.54,150.62
3,150.48,150.47,150.52,150.31,150.04,149.98,151.63,152.48,152.24,152.94,...,150.34,150.30,150.28,150.38,150.41,150.33,150.31,150.31,150.25,150.28
4,150.41,150.35,150.24,150.12,149.87,149.71,151.64,152.37,151.78,152.68,...,150.31,150.20,150.17,150.28,150.31,150.25,150.27,150.22,150.13,150.19


In [59]:
# Checking for missing values:

print("Missing values in PS1:", PS1.isnull().sum().sum())

Missing values in PS1: 0


`PS1` is one of the pressure sensor matrices in the hydraulic dataset. Each row corresponds to one 60-second operating cycle, and each column represents a sampled pressure value within that cycle. Since `PS1` is sampled at 100 Hz, each row contains 6000 measurements.

**Temperature sensor: TS1**

In [58]:
TS1 = pd.read_csv(hydraulics_path + 'TS1.txt', sep='\t', header=None)
print(TS1.shape)
TS1.head()

(2205, 60)


,0,1,2,3,4,5,6,7,8,9,...,50,51,52,53,54,55,56,57,58,59
0,35.570,35.492,35.469,35.422,35.414,35.320,35.227,35.242,35.160,35.176,...,36.008,35.984,35.996,36.039,36.008,36.008,36.094,36.102,36.090,36.152
1,36.156,36.094,35.992,36.008,35.992,35.902,35.824,35.820,35.727,35.727,...,37.328,37.324,37.340,37.332,37.316,37.410,37.418,37.422,37.488,37.477
2,37.488,37.391,37.340,37.312,37.223,37.145,37.059,36.973,36.898,36.879,...,38.457,38.461,38.457,38.469,38.469,38.555,38.527,38.543,38.527,38.621
3,38.633,38.535,38.469,38.379,38.297,38.223,38.125,38.062,37.977,37.969,...,39.441,39.363,39.367,39.457,39.461,39.461,39.473,39.441,39.453,39.461
4,39.461,39.461,39.375,39.281,39.203,39.113,39.043,38.969,38.875,38.883,...,40.324,40.320,40.312,40.340,40.320,40.387,40.391,40.391,40.387,40.391


In [60]:
# Checking for missing values:

print("Missing values in TS1:", TS1.isnull().sum().sum())

Missing values in TS1: 0


`TS1` is one of the temperature sensor matrices in the hydraulic dataset. Each row corresponds to one 60-second operating cycle, and each column represents a sampled temperature value within that cycle. Since `TS1` is sampled at 1 Hz, each row contains 60 measurements.

**Motor power sensor: EPS1**

In [61]:
EPS1 = pd.read_csv(hydraulics_path + 'EPS1.txt', sep='\t', header=None)
print(EPS1.shape)
EPS1.head()

(2205, 6000)


,0,1,2,3,4,5,6,7,8,9,...,5990,5991,5992,5993,5994,5995,5996,5997,5998,5999
0,2411.6,2411.6,2411.6,2411.6,2411.6,2411.6,2411.6,2411.6,2411.6,2409.6,...,2409.6,2409.2,2409.6,2409.4,2409.6,2409.4,2409.6,2409.6,2409.6,2409.6
1,2409.6,2409.6,2409.6,2409.6,2409.6,2409.6,2409.6,2409.6,2409.6,2409.6,...,2398.8,2398.2,2398.2,2398.0,2398.0,2398.0,2398.0,2397.8,2397.8,2397.8
2,2397.8,2397.8,2397.8,2397.8,2397.8,2397.8,2397.8,2397.8,2397.8,2395.8,...,2383.8,2383.8,2383.8,2383.8,2383.8,2383.8,2383.8,2383.8,2383.8,2383.8
3,2383.8,2383.8,2383.8,2383.8,2383.8,2383.8,2383.8,2383.8,2382.8,2382.8,...,2373.2,2372.8,2372.6,2372.4,2372.2,2372.0,2372.0,2372.0,2372.0,2372.0
4,2372.0,2372.0,2372.0,2372.0,2372.0,2372.0,2372.0,2372.0,2372.0,2373.0,...,2370.0,2370.0,2369.8,2369.8,2369.8,2369.8,2369.6,2369.6,2369.6,2369.6


In [62]:
# Checking for missing values:

print("Missing values in EPS1:", EPS1.isnull().sum().sum())

Missing values in EPS1: 0


`EPS1` is the motor power sensor matrix in the hydraulic dataset. Each row corresponds to one 60-second operating cycle, and each column represents a sampled power value within that cycle. Since `EPS1` is sampled at 100 Hz, each row contains 6000 measurements.

**Vibration sensor: VS1**

In [64]:
VS1 = pd.read_csv(hydraulics_path + 'VS1.txt', sep='\t', header=None)

print("VS1 shape:", VS1.shape)
VS1.head()

VS1 shape: (2205, 60)


,0,1,2,3,4,5,6,7,8,9,...,50,51,52,53,54,55,56,57,58,59
0,0.604,0.605,0.611,0.603,0.608,0.608,0.608,0.617,0.619,0.619,...,0.554,0.552,0.545,0.553,0.553,0.539,0.544,0.545,0.535,0.543
1,0.590,0.610,0.626,0.620,0.623,0.619,0.617,0.618,0.619,0.615,...,0.555,0.547,0.548,0.544,0.536,0.542,0.540,0.533,0.531,0.534
2,0.578,0.603,0.638,0.651,0.652,0.662,0.662,0.656,0.652,0.638,...,0.543,0.544,0.543,0.554,0.544,0.544,0.545,0.544,0.530,0.534
3,0.565,0.591,0.608,0.614,0.623,0.645,0.642,0.645,0.642,0.643,...,0.549,0.538,0.553,0.543,0.553,0.555,0.544,0.543,0.543,0.542
4,0.570,0.600,0.623,0.636,0.644,0.642,0.651,0.654,0.660,0.644,...,0.546,0.546,0.544,0.552,0.539,0.540,0.549,0.542,0.533,0.537


In [66]:
# Checking missing values:

print("Missing values in VS1:", VS1.isnull().sum().sum())

Missing values in VS1: 0


`VS1` is the vibration sensor matrix in the hydraulic dataset. Each row corresponds to one 60-second operating cycle, and each column represents a sampled vibration value within that cycle. Since `VS1` is sampled at 1 Hz, each row contains 60 measurements.

**Loading the remaining sensor files**

At this stage, the remaining hydraulic sensor files are loaded in order to build one complete processed dataset for later analysis. Instead of working with each raw matrix separately in the following notebooks, all sensor files will be validated, summarized per cycle, and combined with the target labels from `profile.txt`.

This step is important because the hydraulic dataset is stored in a raw cycle-matrix format, with different sensors sampled at different frequencies. To make the data easier to analyze, each sensor matrix will be reduced to a compact set of summary features for every 60-second cycle.

In [67]:
# Load remaining sensors
FS1 = pd.read_csv(hydraulics_path + 'FS1.txt', sep='\t', header=None)
FS2 = pd.read_csv(hydraulics_path + 'FS2.txt', sep='\t', header=None)
PS2 = pd.read_csv(hydraulics_path + 'PS2.txt', sep='\t', header=None)
PS3 = pd.read_csv(hydraulics_path + 'PS3.txt', sep='\t', header=None)
PS4 = pd.read_csv(hydraulics_path + 'PS4.txt', sep='\t', header=None)
PS5 = pd.read_csv(hydraulics_path + 'PS5.txt', sep='\t', header=None)
PS6 = pd.read_csv(hydraulics_path + 'PS6.txt', sep='\t', header=None)
TS2 = pd.read_csv(hydraulics_path + 'TS2.txt', sep='\t', header=None)
TS3 = pd.read_csv(hydraulics_path + 'TS3.txt', sep='\t', header=None)
TS4 = pd.read_csv(hydraulics_path + 'TS4.txt', sep='\t', header=None)
CE  = pd.read_csv(hydraulics_path + 'CE.txt',  sep='\t', header=None)
CP  = pd.read_csv(hydraulics_path + 'CP.txt',  sep='\t', header=None)
SE  = pd.read_csv(hydraulics_path + 'SE.txt',  sep='\t', header=None)


In [68]:
# Verify all loaded correctly
sensors = {
    'PS1': PS1, 'PS2': PS2, 'PS3': PS3, 'PS4': PS4, 'PS5': PS5, 'PS6': PS6,
    'FS1': FS1, 'FS2': FS2,
    'TS1': TS1, 'TS2': TS2, 'TS3': TS3, 'TS4': TS4,
    'VS1': VS1, 'EPS1': EPS1,
    'CE': CE, 'CP': CP, 'SE': SE
}

for name, df in sensors.items():
    print(f"{name}: shape={df.shape}, missing={df.isnull().sum().sum()}")

PS1: shape=(2205, 6000), missing=0
PS2: shape=(2205, 6000), missing=0
PS3: shape=(2205, 6000), missing=0
PS4: shape=(2205, 6000), missing=0
PS5: shape=(2205, 6000), missing=0
PS6: shape=(2205, 6000), missing=0
FS1: shape=(2205, 600), missing=0
FS2: shape=(2205, 600), missing=0
TS1: shape=(2205, 60), missing=0
TS2: shape=(2205, 60), missing=0
TS3: shape=(2205, 60), missing=0
TS4: shape=(2205, 60), missing=0
VS1: shape=(2205, 60), missing=0
EPS1: shape=(2205, 6000), missing=0
CE: shape=(2205, 60), missing=0
CP: shape=(2205, 60), missing=0
SE: shape=(2205, 60), missing=0


### Aggregating sensor cycles into per-cycle features

To make the hydraulic dataset easier to analyze, each raw sensor matrix is reduced to a compact set of summary statistics for every cycle. This transforms the original high-dimensional cycle data into one tabular feature matrix that can be combined with the condition labels from `profile.txt`.

Each of the four statistics captures a different aspect of sensor behavior within a cycle:

- **`_mean`** - average level of the sensor signal over the cycle (typical operating value)
- **`_std`** - within-cycle variability (how much the sensor fluctuates around its mean; low values suggest stable operation, high values suggest noise, pulsations, or instability)
- **`_min`** - lowest value reached during the cycle
- **`_max`** - highest value reached during the cycle

Together, these four statistics describe both the central tendency and the variability of each sensor within a single operating cycle, while reducing each cycle from a raw time-series vector to a compact four-number representation.

In [70]:
def aggregate_sensor(df, name):
    """Reduce each 60-second cycle matrix to summary statistics per cycle."""
    return pd.DataFrame({
        f'{name}_mean': df.mean(axis=1),
        f'{name}_std':  df.std(axis=1),
        f'{name}_min':  df.min(axis=1),
        f'{name}_max':  df.max(axis=1),
    })


feature_frames = [aggregate_sensor(df, name) for name, df in sensors.items()]

hydraulics_features = pd.concat(
    feature_frames + [profile.reset_index(drop=True)],
    axis=1
)

print(f"Hydraulics feature matrix: {hydraulics_features.shape}")
hydraulics_features.head()

Hydraulics feature matrix: (2205, 73)


,PS1_mean,PS1_std,PS1_min,PS1_max,PS2_mean,PS2_std,PS2_min,PS2_max,PS3_mean,PS3_std,...,CP_max,SE_mean,SE_std,SE_min,SE_max,cooler_condition,valve_condition,pump_leakage,accumulator_pressure,stable_flag
0,160.673492,13.939309,145.83,191.51,109.466914,47.114508,0.0,156.99,1.991475,0.945705,...,2.188,59.157183,23.763984,0.0,79.568,3,100,0,130,1
1,160.603320,14.118967,145.73,191.47,109.354890,47.045611,0.0,157.56,1.976234,0.941967,...,1.414,59.335617,23.857918,0.0,80.441,3,100,0,130,1
2,160.347720,14.192619,145.37,191.41,109.158845,46.992060,0.0,156.97,1.972224,0.943501,...,1.159,59.543150,23.923381,0.0,80.824,3,100,0,130,1
3,160.188088,14.227803,145.14,191.34,109.064807,46.972221,0.0,156.44,1.946575,0.935534,...,1.107,59.794900,24.023005,0.0,80.930,3,100,0,130,1
4,160.000472,14.276434,144.95,191.41,108.931434,46.874946,0.0,158.13,1.922707,0.930335,...,1.106,59.455267,23.972262,0.0,81.100,3,100,0,130,1


In [71]:
hydraulics_features.to_csv('../data/processed/hydraulics_features.csv', index=False)
print("Saved successfully!")

Saved successfully!


The full hydraulic sensor set was loaded and validated. No missing values were detected, and the raw cycle matrices were transformed into a compact per-cycle feature table by extracting summary statistics from each sensor. The processed file is saved for downstream use.

---

## Summary

Both datasets have been loaded, examined structurally, and transformed into analysis-ready forms. The next notebook moves from preprocessing to exploratory data analysis of the processed MetroPT-3 and hydraulic datasets.